# Healthcare Demo: Privacy-Preserving Patient Risk Prediction

This notebook demonstrates FHE-ML for sensitive healthcare data,
showing how hospitals can collaborate on ML models without sharing
patient data in plaintext.

## Use Case
A hospital wants to predict patient readmission risk using a model
trained on data from multiple hospitals, without exposing individual
patient records.

## HIPAA/GDPR Compliance
With FHE, patient data remains encrypted throughout the entire
ML pipeline - from training to inference.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

import sys
sys.path.insert(0, '..')

from sdk.models import (
    LogisticRegression,
    DecisionTreeClassifier,
    ModelConfig,
    TreeConfig,
)

## 1. Synthetic Patient Data

We generate synthetic healthcare data simulating patient records
with features relevant to readmission risk prediction.

In [ ]:
def generate_patient_data(n_patients=1000, random_state=42):
    """Generate synthetic patient data for readmission prediction."""
    np.random.seed(random_state)
    
    # Demographics
    age = np.random.normal(65, 15, n_patients).clip(18, 95)
    gender = np.random.binomial(1, 0.48, n_patients)  # 0=F, 1=M
    
    # Clinical indicators
    length_of_stay = np.random.exponential(5, n_patients).clip(1, 30)
    num_medications = np.random.poisson(8, n_patients).clip(0, 25)
    num_procedures = np.random.poisson(2, n_patients).clip(0, 10)
    num_diagnoses = np.random.poisson(5, n_patients).clip(1, 15)
    
    # Lab values (normalized)
    glucose = np.random.normal(120, 40, n_patients).clip(50, 300)
    hemoglobin = np.random.normal(13, 2, n_patients).clip(7, 18)
    creatinine = np.random.exponential(1.2, n_patients).clip(0.5, 8)
    
    # Prior history
    prior_admissions = np.random.poisson(1.5, n_patients).clip(0, 10)
    has_diabetes = np.random.binomial(1, 0.25, n_patients)
    has_heart_disease = np.random.binomial(1, 0.20, n_patients)
    
    # Generate readmission risk (target)
    # Higher risk with: older age, longer stay, more meds, prior admissions
    risk_score = (
        0.02 * (age - 50) +
        0.05 * length_of_stay +
        0.03 * num_medications +
        0.1 * prior_admissions +
        0.5 * has_diabetes +
        0.6 * has_heart_disease +
        0.01 * (glucose - 100) +
        0.02 * creatinine +
        np.random.normal(0, 0.5, n_patients)
    )
    
    # Convert to binary outcome
    readmitted = (risk_score > np.percentile(risk_score, 70)).astype(int)
    
    # Create DataFrame
    df = pd.DataFrame({
        'age': age,
        'gender': gender,
        'length_of_stay': length_of_stay,
        'num_medications': num_medications,
        'num_procedures': num_procedures,
        'num_diagnoses': num_diagnoses,
        'glucose': glucose,
        'hemoglobin': hemoglobin,
        'creatinine': creatinine,
        'prior_admissions': prior_admissions,
        'has_diabetes': has_diabetes,
        'has_heart_disease': has_heart_disease,
        'readmitted': readmitted,
    })
    
    return df

# Generate data
patients = generate_patient_data(n_patients=1000)
print(f"Generated {len(patients)} patient records")
print(f"\nReadmission rate: {patients['readmitted'].mean():.1%}")
patients.head()

In [ ]:
# Summary statistics
print("Dataset Statistics:")
patients.describe().round(2)

## 2. Data Preparation

Prepare features for FHE-compatible training.

In [ ]:
# Separate features and target
feature_cols = [
    'age', 'gender', 'length_of_stay', 'num_medications',
    'num_procedures', 'num_diagnoses', 'glucose', 'hemoglobin',
    'creatinine', 'prior_admissions', 'has_diabetes', 'has_heart_disease'
]

X = patients[feature_cols].values
y = patients['readmitted'].values

# Normalize features (critical for FHE)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} patients")
print(f"Test set: {len(X_test)} patients")
print(f"Features: {len(feature_cols)}")
print(f"\nClass distribution (train): {np.bincount(y_train)}")

## 3. Logistic Regression for Risk Prediction

In [ ]:
# Train logistic regression
lr_config = ModelConfig(
    learning_rate=0.1,
    n_epochs=200,
    verbose=False,
)

lr_model = LogisticRegression(config=lr_config)
lr_model._fit_plaintext(X_train, y_train)

# Predictions
y_pred_lr = lr_model._predict_plaintext(X_test)
y_pred_lr_binary = (y_pred_lr > 0.5).astype(int)

print("Logistic Regression Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_lr_binary):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_lr_binary):.4f}")
print(f"  Recall: {recall_score(y_test, y_pred_lr_binary):.4f}")
print(f"  F1 Score: {f1_score(y_test, y_pred_lr_binary):.4f}")

In [ ]:
# Feature importance (coefficients)
print("\nRisk Factors (Logistic Regression Coefficients):")
print("Positive = increases readmission risk")
print("-" * 50)

coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr_model.weights
}).sort_values('Coefficient', ascending=False)

for _, row in coef_df.iterrows():
    sign = '+' if row['Coefficient'] > 0 else ''
    print(f"  {row['Feature']:<20} {sign}{row['Coefficient']:.4f}")

## 4. Decision Tree Classifier

In [ ]:
# Train decision tree
tree_config = TreeConfig(
    max_depth=4,
    learning_rate=0.1,
    n_epochs=50,
)

tree_model = DecisionTreeClassifier(config=tree_config)
tree_model._fit_plaintext(X_train, y_train)

# Predictions
y_pred_tree = tree_model._predict_plaintext(X_test)

print("Decision Tree Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_tree):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_tree):.4f}")
print(f"  Recall: {recall_score(y_test, y_pred_tree):.4f}")
print(f"  F1 Score: {f1_score(y_test, y_pred_tree):.4f}")

## 5. Model Comparison

In [ ]:
# Compare models
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr_binary),
        accuracy_score(y_test, y_pred_tree)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr_binary),
        precision_score(y_test, y_pred_tree)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr_binary),
        recall_score(y_test, y_pred_tree)
    ],
    'F1': [
        f1_score(y_test, y_pred_lr_binary),
        f1_score(y_test, y_pred_tree)
    ],
})

print("Model Comparison:")
print(results.to_string(index=False))

## 6. Privacy-Preserving Workflow

In a real deployment, the workflow would be:

```
Hospital A                    ML Service                    Hospital B
    |                             |                             |
    |-- Encrypt patient data ---> |                             |
    |                             | <-- Encrypt patient data ---|  
    |                             |                             |
    |                       [Train on encrypted]                |
    |                             |                             |
    |<-- Encrypted predictions ---|--- Encrypted predictions -->|
    |                             |                             |
    |   [Decrypt locally]         |         [Decrypt locally]   |
```

Key privacy guarantees:
- Patient data never leaves the hospital in plaintext
- ML service only sees encrypted ciphertexts
- Only data owners can decrypt predictions

In [ ]:
# Simulate single patient prediction
def predict_patient_risk(patient_data, model):
    """
    In production, this would:
    1. Encrypt patient_data
    2. Send encrypted data to model
    3. Receive encrypted prediction
    4. Decrypt locally
    """
    # Normalize
    patient_scaled = scaler.transform(patient_data.reshape(1, -1))
    
    # Predict
    risk_score = model._predict_plaintext(patient_scaled)[0]
    
    return risk_score

# Example patient
example_patient = np.array([
    75,    # age
    1,     # gender (male)
    8,     # length_of_stay
    12,    # num_medications
    3,     # num_procedures
    6,     # num_diagnoses
    180,   # glucose
    11,    # hemoglobin
    2.1,   # creatinine
    3,     # prior_admissions
    1,     # has_diabetes
    1,     # has_heart_disease
])

risk = predict_patient_risk(example_patient, lr_model)

print("Example Patient Risk Assessment:")
print(f"  Age: {example_patient[0]:.0f}")
print(f"  Diabetes: {'Yes' if example_patient[10] else 'No'}")
print(f"  Heart Disease: {'Yes' if example_patient[11] else 'No'}")
print(f"  Prior Admissions: {example_patient[9]:.0f}")
print(f"\n  Readmission Risk Score: {risk:.2%}")
print(f"  Risk Level: {'HIGH' if risk > 0.5 else 'LOW'}")

## 7. Compliance Benefits

### HIPAA Compliance
- Protected Health Information (PHI) remains encrypted
- No data sharing in plaintext between organizations
- Audit trail via blockchain (optional)

### GDPR Compliance
- Data minimization: only encrypted data processed
- Privacy by design: FHE built into the architecture
- Right to erasure: delete encryption keys

### LGPD (Brazil) Compliance
- Similar protections to GDPR
- Demonstrates "adequate security measures"

## Summary

This demo showed how FHE-ML enables:

1. **Privacy-preserving training** on sensitive healthcare data
2. **Risk prediction** without exposing patient records
3. **Multi-hospital collaboration** with encrypted data
4. **Regulatory compliance** (HIPAA, GDPR, LGPD)

### Production Considerations
- Use real clinical datasets (with proper IRB approval)
- Implement proper key management
- Add differential privacy for additional protection
- Use blockchain for audit trail